In [1]:
!pip install yfinance

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.2/948.2 kB 9.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 9.5 MB/s eta 0:00:00ta 0:00:01
  Created wheel for peewee: filename=peewee-3.17.8-cp311-cp311-linux_aarch64.whl size=312276 sha256=b8dccc05d1942dc8184ffe66a39d6a8907f1fb56d0fa313a0ba9f97bd8ebd053
  Stored in directory: /home/jovyan/.cache/pip/wheels/ff/6c/15/506e25bc390de450a7fa53c155cd9b0fbd13ad3e84a9abc183
Successfully built peewee


In [3]:
!python -V

Python 3.11.6


In [16]:
import yfinance as yf

dat = yf.Ticker("MSFT")
df = dat.history(period="6mo", interval="1d")
df

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2024-06-07 00:00:00-04:00,424.583341,424.663024,421.395467,422.242249,13621700,0.0,0.0
2024-06-10 00:00:00-04:00,423.089020,426.456174,422.282095,426.246979,14003000,0.0,0.0
2024-06-11 00:00:00-04:00,423.866056,431.178210,423.636918,431.038727,14551100,0.0,0.0
2024-06-12 00:00:00-04:00,433.668746,441.718083,431.606590,439.386963,22366200,0.0,0.0
2024-06-13 00:00:00-04:00,439.177787,441.708161,437.703390,439.904999,15960600,0.0,0.0
...,...,...,...,...,...,...,...
2024-12-02 00:00:00-05:00,421.570007,433.000000,421.309998,430.980011,20207200,0.0,0.0
2024-12-03 00:00:00-05:00,429.839996,432.470001,427.739990,431.200012,18302000,0.0,0.0
2024-12-04 00:00:00-05:00,433.029999,439.670013,432.630005,437.420013,26009400,0.0,0.0


In [17]:
# Convert the index to datetime.date
df.index = df.index.date
df.iloc[0].name

datetime.date(2024, 6, 7)

In [19]:
from datetime import date

# Remove today's date
today_date = date.today()
df = df[df.index != today_date]
df.iloc[len(df)-1]

Open            4.423000e+02
High            4.461000e+02
Low             4.417700e+02
Close           4.435700e+02
Volume          1.881440e+07
Dividends       0.000000e+00
Stock Splits    0.000000e+00
Name: 2024-12-06, dtype: float64

In [27]:
prices = df.loc[:, 'Close'].to_numpy()
dates = df.index.to_numpy()

print(prices[0], dates[0])

422.24224853515625 2024-06-07


In [36]:
import numpy as np

last_checkpoint_at = date.fromisoformat('2024-12-03')

last_checkpoint_index = np.where(dates == last_checkpoint_at)
print(last_checkpoint_index[0][0], len(dates))

WINDOW_SIZE = 10
FRAME_BOUND = (last_checkpoint_index[0][0], len(dates) - 1)

123 127


In [38]:
# Z-score normalization function
def z_score(values):
    mean = np.mean(values)
    std_dev = np.std(values)
    if std_dev == 0:
        return np.zeros_like(values)
    return ((values - mean) / std_dev) / 3 # Divide by 3 to keep values between -1 and 1

In [54]:
import pandas_ta as ta

ma1 = df.ta.ema(length=20).to_numpy()
ma1 = np.where(np.isfinite(ma1), ma1, 0)
#ma1 = ma1[FRAME_BOUND[0] - WINDOW_SIZE: FRAME_BOUND[1]]
ma1_z = z_score(ma1)



rsi = df.ta.rsi().to_numpy()
rsi = np.where(np.isfinite(rsi), rsi, 0)
#rsi = rsi[self.frame_bound[0] - self.window_size: self.frame_bound[1]]
rsi_z = z_score(rsi)

print(
    dates[FRAME_BOUND[0]:],
    ma1_z[FRAME_BOUND[0]:],
    rsi_z[FRAME_BOUND[0]:FRAME_BOUND[1]],
)

[datetime.date(2024, 12, 3) datetime.date(2024, 12, 4)
 datetime.date(2024, 12, 5) datetime.date(2024, 12, 6)] [0.13474159 0.13800372 0.14204452 0.1458995 ] [0.26776967 0.3442277  0.40014411]


In [55]:
feat = np.column_stack((ma1_z, rsi_z))
print(feat[0], prices[0], dates[0])

[-0.79317706 -0.77715942] 422.24224853515625 2024-06-07
